# himaTTS Hindi on ONNX Runtime, CPU only
Use a CPU runtime. Upload `hima-tts-hi-onnx.zip` in step 2 (and optionally `hima-tts-hi-onnx-fp32-extras.zip` for the A/B test in step 7).

## 1. Install

In [ ]:
!pip -q install onnxruntime numpy sentencepiece safetensors

## 2. Get the files
Option A: upload. Option B: put the zip(s) in My Drive and set `USE_DRIVE = True`.

In [ ]:
import os, zipfile, glob
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive"          # folder holding the zip(s) when USE_DRIVE = True
if not os.path.exists("/content/hima-tts-hi-onnx"):
    if USE_DRIVE:
        from google.colab import drive; drive.mount("/content/drive"); zips = glob.glob(DRIVE_DIR + "/hima-tts-hi-onnx*.zip")
    else:
        from google.colab import files; zips = list(files.upload().keys())
    for z in zips:
        with zipfile.ZipFile(z) as f: f.extractall("/content")
%cd /content/hima-tts-hi-onnx
!ls models/lm

## 3. Speak

In [ ]:
import time, numpy as np
from IPython.display import Audio, display
from hima_onnx import Hima, SR
h = Hima(lm="w8")
print("voices:", h.voices)
TEXT = "यार, आज office में इतना काम था कि lunch करने का भी time नहीं मिला।"
t = time.time(); y = h.say(TEXT, voice="Meera", seed=1); dt = time.time() - t
print(f"{len(y)/SR:.2f}s audio in {dt:.2f}s (real-time factor {dt/(len(y)/SR):.2f}; the very first call also includes warm-up)")
display(Audio(y, rate=SR))

## 4. Streaming (time to first audio)

In [ ]:
t0 = time.time(); chunks = []; first = None
for c in h.stream(TEXT, voice="Ishaan", seed=1):
    if first is None: first = time.time() - t0
    chunks.append(c)
print(f"first audio after {first*1000:.0f} ms, {len(chunks)} chunks")
display(Audio(np.concatenate(chunks), rate=SR))

## 5. All built-in voices

In [ ]:
for v in h.voices:
    print(v); display(Audio(h.say("नमस्ते, आज मौसम बहुत अच्छा है।", voice=v, seed=0), rate=SR))

## 6. Random voice (stays the same across sentences) and saving a voice

In [ ]:
voice = h.make_random_voice(seed=3)                 # change the seed for a different voice
display(Audio(h.say("इस किताब की कीमत ₹2,500 है। यह 15/08/2026 को प्रकाशित हुई थी।", voice=voice, seed=0), rate=SR))
np.savez("my_voice.npz", **voice)                   # reuse later with voice="my_voice.npz"

## 7. A/B: 8-bit vs full precision (needs the fp32 extras zip)
Same text, voice and seed. Sampling is random so small differences are normal; judge by listening.

In [ ]:
import os
if os.path.exists("models/lm/lm_voices_fp32.onnx"):
    for kind in ["w8", "fp32"]:
        q = Hima(lm=kind); print(kind); display(Audio(q.say(TEXT, voice="Meera", seed=1), rate=SR))
else:
    print("fp32 files not found: upload hima-tts-hi-onnx-fp32-extras.zip and re-run step 2")

## 8. Your own text

In [ ]:
MY_TEXT = "यहाँ अपना वाक्य लिखिए।"
y = h.say(MY_TEXT, voice="Nandini", seed=0, temperature=0.3)
display(Audio(y, rate=SR))
import wave
with wave.open("out.wav", "wb") as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR); w.writeframes((np.clip(y, -1, 1) * 32767).astype("<i2").tobytes())